In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split    

In [2]:
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\markjake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\markjake\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\markjake\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
with open("../data/theverdict.txt", "r") as file:
    raw_text = file.read()

In [4]:
print(f"raw_text len: {len(raw_text)}")

raw_text len: 20479


## Tokenization and vocabulary

In [5]:
sentences = sent_tokenize(raw_text.lower())

In [6]:
tokenized_sentence = [
    word_tokenize(word)
    for word in sentences
]

In [7]:
print(f"sentences size: {len(sentences)}")
print(f"tokenized_sentence size: {len(tokenized_sentence)}")

sentences size: 235
tokenized_sentence size: 235


In [8]:
embedding_model = Word2Vec(sentences=tokenized_sentence, vector_size=100, window=5, min_count=1, sg=1, workers=4, epochs=10)

In [38]:
len(embedding_model.wv.index_to_key)

1080

In [10]:
embedding_model.wv.index_to_key

[',',
 '.',
 'the',
 'i',
 'of',
 'to',
 '--',
 'and',
 'it',
 'a',
 'he',
 '``',
 'was',
 "''",
 'his',
 'that',
 'had',
 'my',
 'in',
 "'s",
 'me',
 'him',
 'with',
 "n't",
 'she',
 'you',
 'on',
 'have',
 'as',
 '!',
 '?',
 'one',
 'but',
 'at',
 ';',
 'mrs.',
 ':',
 'her',
 'up',
 'gisburn',
 'for',
 'been',
 'stroud',
 'not',
 'by',
 'so',
 "'",
 'jack',
 'said',
 'when',
 'could',
 'just',
 'if',
 'only',
 'all',
 'see',
 'out',
 'say',
 'an',
 'would',
 'about',
 'did',
 'be',
 'picture',
 'what',
 'there',
 'they',
 'or',
 'know',
 'down',
 'through',
 'is',
 'do',
 'himself',
 'painting',
 'no',
 'like',
 'eyes',
 'were',
 'thought',
 'knew',
 'how',
 'oh',
 'now',
 'from',
 'put',
 'back',
 'because',
 'such',
 'own',
 'poor',
 'again',
 'who',
 'rickham',
 'thing',
 'donkey',
 'told',
 'never',
 'paint',
 'painted',
 'work',
 'first',
 'pictures',
 'their',
 'between',
 'into',
 'too',
 'given',
 'little',
 'fact',
 'felt',
 'well',
 'course',
 'last',
 'rather',
 'always',


In [11]:
# word vector arithmetic

result = embedding_model.wv.most_similar(positive=["strongest", "dead"], topn=5)

In [12]:
result 

[('been', 0.9978092908859253),
 ('off', 0.9978071451187134),
 ('.', 0.9977976083755493),
 ('it', 0.9977927803993225),
 ('head', 0.9977763891220093)]

In [13]:
def create_sequence(sentences):
    pairs = []

    for sentence in sentences:
        for i in range(1, len(sentence)):
            context = sentence[:i]
            target = sentence[i]

            pairs.append((context, target))

    return pairs

In [14]:
def context_vector(context, embedding_model):
    vectors = [
        embedding_model.wv[word]
        for word in context
        if word in embedding_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)
    
    return np.mean(vectors, axis=0)

In [15]:
pairs = create_sequence(tokenized_sentence)

In [16]:
X = np.array([
    context_vector(context, embedding_model)
    for context, target in pairs
])
y = np.array([
    target
    for _, target in pairs
])
print(X.shape)
print(y.shape)

(4316, 100)
(4316,)


In [17]:
word_to_idx = {
    word: idx
    for idx, word in enumerate(embedding_model.wv.index_to_key)
}

y = np.array([
    word_to_idx[word]
    for word in y
])

In [18]:
idx_to_word = {
    idx: word
    for word, idx in word_to_idx.items()
}

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
print(X_train.shape)
print(y_train.shape)

(3452, 100)
(3452,)


In [21]:
# initialize parameters
def initialize_parameters(input_size, hidden_size1, hidden_size2, output_size):
    np.random.seed(42)
    parameters = {
        "w1": np.random.randn(hidden_size1, input_size) * 0.01,
        "b1": np.zeros((hidden_size1, 1)),
        "w2": np.random.randn(hidden_size2, hidden_size1) * 0.01,
        "b2": np.zeros((hidden_size2, 1)),
        "w3": np.random.randn(output_size, hidden_size2) * 0.01,
        "b3": np.zeros((output_size, 1))
       
    }
    return parameters

In [22]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

In [23]:
def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

In [24]:
def relu(Z):
    return np.maximum(0, Z)

In [25]:
Z = np.array([0.1, 0.9, 0.3, 0.5])

softmax(Z)

array([0.16838508, 0.37474789, 0.205666  , 0.25120102])

In [26]:
def forward_propagation(X, paramameters):
    w1, b1, w2, b2, w3, b3 = paramameters["w1"], paramameters["b1"], paramameters["w2"], paramameters["b2"], paramameters["w3"], paramameters["b3"]
    # first layer
    Z1 = np.dot(w1, X) + b1
    A1 = relu(Z1)

    # second layer
    Z2 = np.dot(w2, A1) + b2
    A2 = relu(Z2)

    # output layer
    Z3 = np.dot(w3, A2) + b3
    
    # softmax
    A3 = softmax(Z3)

    return A1, A2, A3
    

In [27]:
def compute_cost(Y, A3):
    m = Y.shape[0]

    correct_probs = A3[np.arange(m), Y]
    L = -1 / m * np.sum(np.log(correct_probs))

    return L

In [28]:
input_size = embedding_model.wv.vector_size
hiddenLayer1_size = 12
hiddenLayer2_size = 6
vocab_size = len(embedding_model.wv.index_to_key)
output_size = vocab_size

In [29]:
print("Total vocabulary:", vocab_size)
print("Training samples:", len(y_train))
print("Unique training targets:", len(np.unique(y_train)))
print("Unique test targets:", len(np.unique(y_test)))

Total vocabulary: 1080
Training samples: 3452
Unique training targets: 934
Unique test targets: 361


In [30]:
print(f"input_size: {input_size}")
print(f"hiddenLayer1_size: {hiddenLayer1_size}")
print(f"hiddenLayer2_size: {hiddenLayer2_size}")
print(f"vocab_size: {vocab_size}")
print(f"output_size: {output_size}")

input_size: 100
hiddenLayer1_size: 12
hiddenLayer2_size: 6
vocab_size: 1080
output_size: 1080


In [31]:
params = initialize_parameters(input_size, hiddenLayer1_size, hiddenLayer2_size, output_size)

In [32]:
for key, val in params.items():
    print(f"{key} shape: {val.shape}")

print(f"X shape: {X.shape}")

w1 shape: (12, 100)
b1 shape: (12, 1)
w2 shape: (6, 12)
b2 shape: (6, 1)
w3 shape: (1080, 6)
b3 shape: (1080, 1)
X shape: (4316, 100)


In [33]:
A1, A2, A3 = forward_propagation(X_train, params)
cost = compute_cost(y_train, A3)

ValueError: shapes (12,100) and (3452,100) not aligned: 100 (dim 1) != 3452 (dim 0)

In [93]:
print("Output shape:", A3.shape)
print("Minimum target index:", y_train.min())
print("Maximum target index:", y_train.max())
print("Vocabulary size:", vocab_size)

Output shape: (1080, 3452)
Minimum target index: 0
Maximum target index: 1079
Vocabulary size: 1080


In [80]:
for epoch in range(5):
    A1, A2, A3 = forward_propagation(X, params)
    cost = compute_cost(y, A3)

    # Backpropagation goes here
    # Update parameters goes here

    print(f"Epoch {epoch + 1}: loss = {cost:.4f}")

IndexError: index 1080 is out of bounds for axis 0 with size 1080

In [40]:
test_contexts = [
    ["say"],
    ["the"],
    ["court"],
    ["i", "don't"],
    ["verdict"]
]

for context in test_contexts:
    x = context_vector(context, embedding_model).reshape(1, -1)

    _, _, probabilities = forward_propagation(x, params)

    predicted_idx = np.argmax(probabilities[:, 0])

    print(
        f"{' '.join(context)} -> "
        f"{idx_to_word[predicted_idx]} "
        f"({probabilities[predicted_idx, 0]:.6f})"
    )


say -> say (0.000926)
the -> say (0.000926)
court -> , (0.000926)
i don't -> say (0.000926)
verdict -> , (0.000926)


In [189]:
print("X:", X.shape)
print("y:", y.shape)
print("A3:", A3.shape)
print("W1:", params["w1"].shape)
print("W2:", params["w2"].shape)
print("W3:", params["w3"].shape)

X: (4316, 100)
y: (4316,)
A3: (1080, 4316)
W1: (12, 100)
W2: (6, 12)
W3: (1080, 6)


In [166]:
print("Input shape:", X.shape)
print("Layer 1 activation shape:", A1.shape)
print("Layer 2 activation shape:", A2.shape)
print("Output shape:", A3.shape)

print("Expected output shape:", (vocab_size, X.shape[0]))

Input shape: (4316, 100)
Layer 1 activation shape: (12, 4316)
Layer 2 activation shape: (6, 4316)
Output shape: (1080, 4316)
Expected output shape: (1080, 4316)
